# Module 4 Lab — Statistics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/applied-data-science-python/blob/main/module-04-statistical-analysis/notebooks/Module04_Statistics_Lab.ipynb)

**Applied Data Science with Python** · Open Educational Resource · CC BY 4.0

Each section follows the same path: **Learning Objectives → Concept → Guided Practice → Your Turn**. Run the cells in order. In *Your Turn* cells, replace every `___` (or the `# YOUR CODE HERE` line) and run the cell; the `assert` checks tell you when you are right.

Estimated time: 9.5 hours of practical work.

In [ ]:
# Run this cell first. It loads course datasets from the repository
# (works locally, in JupyterLab, and in Google Colab).
import os
import pandas as pd

REPO_RAW = "https://raw.githubusercontent.com/YOUR-GITHUB-USERNAME/applied-data-science-python/main/datasets/"   # <- replace YOUR-GITHUB-USERNAME once you publish the repo

def load_data(name, **kwargs):
    """Load a course dataset by file name, e.g. load_data("titanic.csv")."""
    for folder in ("../../datasets", "../datasets", "datasets", "."):
        path = os.path.join(folder, name)
        if os.path.exists(path):
            return pd.read_csv(path, **kwargs)
    return pd.read_csv(REPO_RAW + name, **kwargs)

## Part 1 — Descriptive statistics

### Learning Objectives
- Summarise centre, spread and shape
- Choose mean vs median sensibly

### Concept
Centre: mean, median, mode. Spread: range, variance, standard deviation, IQR. Shape: skewness and kurtosis. For skewed data, report the median and IQR.

### Guided Practice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

housing = load_data("california_housing.csv")
housing[["median_income", "median_house_value", "housing_median_age"]].describe().round(2)

### Your Turn
**Exercise 1.1** — For `median_house_value`, compute the mean, median, standard deviation, IQR and skewness. Store them in a dict `summary` with keys `mean`, `median`, `std`, `iqr`, `skew`.

In [ ]:
v = housing["median_house_value"]
summary = {
    "mean": ___,
    "median": ___,
    "std": ___,
    "iqr": ___,
    "skew": ___,
}
assert summary["mean"] > summary["median"] and summary["skew"] > 0
assert abs(summary["iqr"] - (v.quantile(0.75) - v.quantile(0.25))) < 1
print({k: round(x, 2) for k, x in summary.items()})

**Exercise 1.2** — Using Iris `petal_width` for **setosa only**, check the empirical rule: compute the share of values within 1, 2 and 3 standard deviations (`shares`, a list of 3 floats). Is setosa petal width close to normal? Answer in a comment.

In [ ]:
iris = load_data("iris.csv")
pw = ___
z = ___
shares = ___
assert len(shares) == 3 and shares[0] <= shares[1] <= shares[2]
print([round(s, 2) for s in shares])

## Part 2 — Inference: sampling, confidence intervals, correlation

### Learning Objectives
- See the Central Limit Theorem in action
- Build and interpret confidence intervals
- Compute and interpret correlations

### Concept
A 95% confidence interval for a mean is `mean ± t* × SE`, where SE = s/√n. In SciPy: `stats.t.interval(0.95, df=n-1, loc=mean, scale=stats.sem(sample))`.

### Guided Practice

In [ ]:
pop = housing["population"]
rng = np.random.default_rng(42)
for n in (5, 30, 200):
    means = [pop.sample(n, random_state=rng).mean() for _ in range(1000)]
    plt.hist(means, bins=40, alpha=0.5, label=f"n={n}")
plt.legend(); plt.title("Sampling distribution of the mean gets narrower and more normal")
plt.show()

### Your Turn
**Exercise 2.1** — Draw a random sample of 100 districts (`random_state=7`) and compute a 95% CI for mean `median_income` (`lo`, `hi`). Does it contain the population mean?

In [ ]:
sample = ___
lo, hi = ___
contains = lo <= housing["median_income"].mean() <= hi

assert lo < sample["median_income"].mean() < hi
print(f"95% CI: {lo:.2f} to {hi:.2f}; contains population mean: {contains}")

**Exercise 2.2** — Compute Pearson and Spearman correlation matrices for the numeric housing columns, find the variable most correlated with `median_house_value` (excluding itself) into `best_predictor`, and draw a heatmap.

In [ ]:
num = housing.select_dtypes("number")
pearson = ___
spearman = ___
best_predictor = ___

assert best_predictor == "median_income"
print(best_predictor)

## Part 3 — Hypothesis testing

### Learning Objectives
- Write H₀ and H₁ for a question
- Choose and run the right SciPy test
- Report the result in a sentence

### Concept
| Question | Test |
|---|---|
| One mean vs a value | `stats.ttest_1samp` |
| Two independent groups | `stats.ttest_ind` (use `equal_var=False`) |
| Same subjects twice | `stats.ttest_rel` |
| Two categorical variables | `stats.chi2_contingency` |
| Three or more group means | `stats.f_oneway` |

### Your Turn
**Exercise 3.1** — *Did first-class passengers on the Titanic have a different mean age from third-class passengers?* Write H₀/H₁ in the markdown cell below, then run a Welch t-test and store the p-value in `p_age`.

*H₀:* …  
*H₁:* …

In [ ]:
titanic = load_data("titanic.csv")
first = ___
third = ___
t_stat, p_age = ___

assert p_age < 0.05
print(f"t = {t_stat:.2f}, p = {p_age:.2e}")

**Exercise 3.2** — Is survival independent of passenger class? Build the contingency table and run a chi-square test (`p_class`).

In [ ]:
table = ___
chi2, p_class, dof, expected = ___

assert dof == 2 and p_class < 0.05
print(table); print(f"chi2 = {chi2:.1f}, p = {p_class:.2e}")

**Exercise 3.3** — Does average `revenue` per transaction differ across the five retail categories? Run a one-way ANOVA (`p_anova`), then show the group means. Also run a paired t-test on the supplied before/after data (`p_paired`).

In [ ]:
sales = load_data("retail_sales.csv")
groups = ___
F, p_anova = ___

rng = np.random.default_rng(11)
before = rng.normal(70, 10, 30)
after = before + rng.normal(3, 4, 30)
t2, p_paired = ___

assert p_anova < 0.05 and p_paired < 0.05
print(f"ANOVA p = {p_anova:.2e}; paired p = {p_paired:.2e}")

## Part 4 — Regression with statsmodels

### Learning Objectives
- Fit and read an OLS model
- Check residuals
- Explain a coefficient in plain language

### Concept
`model = sm.OLS(y, sm.add_constant(x)).fit()` then `model.summary()`. Key numbers: `params` (intercept and slope), `pvalues`, `rsquared`.

### Your Turn
**Exercise 4.1** — Regress `median_house_value` on `median_income`. Store the slope in `slope` and R² in `r2`, and print the summary.

In [ ]:
X = ___
y = ___
model = ___
slope = ___
r2 = ___

assert 40000 < slope < 43000 and 0.45 < r2 < 0.49
print(model.summary())

**Exercise 4.2** — Plot residuals against fitted values. Then, in a markdown cell, write **two sentences for a city council member** explaining what the model says and one limitation you can see in the residual plot.

In [ ]:
# YOUR CODE HERE

## Mini-project — A statistical brief

Pick **one** question about the retail data (for example: *Do members buy more units per transaction than guests?* or *Is rating related to discount?*). In under one page of notebook:
1. Describe the relevant variables (centre, spread, a chart).
2. State H₀/H₁, choose a test, and justify it.
3. Run it and report the result with an effect size (difference in means or correlation).
4. Write a 3-sentence conclusion for a manager.

Then take the **Module 4 quiz**.

In [ ]:
# Mini-project: your analysis here